In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/comment-category-prediction-challenge/Sample.csv
/kaggle/input/comment-category-prediction-challenge/train.csv
/kaggle/input/comment-category-prediction-challenge/test.csv


In [2]:
# Basic libraries
import numpy as np
import pandas as pd

# Visualization (optional but useful)
import matplotlib.pyplot as plt
import seaborn as sns

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


# ======================
# 1. IMPORT LIBRARIES
# ======================
import numpy as np
import pandas as pd
import os

# Visualization (optional)
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print("Libraries Imported Successfully")


Libraries Imported Successfully


In [3]:
import pandas as pd

train_df = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/train.csv')
test_df  = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/test.csv')

train_df.head()

,created_date,post_id,emoticon_1,emoticon_2,emoticon_3,upvote,downvote,if_1,if_2,race,religion,gender,disability,comment,label
0,2024-01-18 08:43:57.397508+00:00,73,0,0,0,0,1,0,10,NaN,NaN,NaN,False,She might be a bright spot for a party keou on...,2
1,2024-03-24 21:43:11.490017+00:00,39,0,0,0,6,0,0,4,NaN,NaN,NaN,False,"Under Alaska law, a non-tribal member is not b...",0
2,2024-04-24 20:32:17.014931+00:00,31,0,1,1,0,0,0,10,NaN,NaN,NaN,False,in the future please spare me your strawman dr...,2
3,2023-05-28 22:00:14.214527+00:00,39,0,0,0,5,0,0,10,NaN,NaN,NaN,False,"PS: That should have been ""rot"" instead of ""co...",2
4,2023-09-09 23:12:05.689498+00:00,39,0,0,0,0,0,0,10,NaN,NaN,NaN,False,"Today, the confederate flag...tomorrow, the na...",2


In [4]:
train_df.shape

(198000, 15)

In [5]:
test_df.shape[1]

14

In [6]:
train_df.select_dtypes(include='object').columns
len(train_df.select_dtypes(include='object').columns)

5

In [7]:
train_df.select_dtypes(include=['int64','float64']).columns
len(train_df.select_dtypes(include=['int64','float64']).columns)

9

In [8]:
train_df.dtypes

created_date    object
post_id          int64
emoticon_1       int64
emoticon_2       int64
emoticon_3       int64
upvote           int64
downvote         int64
if_1             int64
if_2             int64
race            object
religion        object
gender          object
disability        bool
comment         object
label            int64
dtype: object

In [9]:
train_df.isnull().sum()

created_date         0
post_id              0
emoticon_1           0
emoticon_2           0
emoticon_3           0
upvote               0
downvote             0
if_1                 0
if_2                 0
race            145423
religion        145423
gender          145423
disability           0
comment              1
label                0
dtype: int64

In [10]:
train_df['label'].nunique()

4

In [11]:
train_df['label'].value_counts(normalize=True) * 100

label
0    57.663131
2    31.535354
1     8.039394
3     2.762121
Name: proportion, dtype: float64

In [12]:
train_df['upvote'].median()

1.0

In [13]:
train_df[['upvote','downvote','if_1','if_2']].max()

upvote       201
downvote     107
if_1        1860
if_2        1833
dtype: int64

In [14]:
train_df['if_2'].min()

3

In [15]:
cols = ['gender','race','religion','disability']

for col in cols:
    train_df[col] = train_df[col].astype(str).str.lower().str.strip()
    test_df[col]  = test_df[col].astype(str).str.lower().str.strip()

    # Fix spelling
    train_df[col] = train_df[col].replace('unkown', 'unknown')
    test_df[col]  = test_df[col].replace('unkown', 'unknown')

    # Fill NaN
    train_df[col] = train_df[col].replace('nan', 'unknown')
    test_df[col]  = test_df[col].replace('nan', 'unknown')


In [16]:
from sklearn.preprocessing import LabelEncoder

for col in cols:
    le = LabelEncoder()
    
    # Fit on BOTH train + test categories
    le.fit(list(train_df[col]) + list(test_df[col]))
    
    train_df[col] = le.transform(train_df[col])
    test_df[col]  = le.transform(test_df[col])


In [17]:
train_df['comment'].isnull().sum()
test_df['comment'].isnull().sum()

np.int64(0)

In [18]:
train_df['comment'] = train_df['comment'].fillna('')
test_df['comment']  = test_df['comment'].fillna('')

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000)

X_text_train = tfidf.fit_transform(train_df['comment'])
X_text_test  = tfidf.transform(test_df['comment'])

In [20]:
num_cols = ['upvote','downvote','if_1','if_2','gender','race','religion','disability']

X_num_train = train_df[num_cols].values
X_num_test = test_df[num_cols].values

from scipy.sparse import hstack

X_train = hstack((X_text_train, X_num_train))
X_test = hstack((X_text_test, X_num_test))

y_train = train_df['label']

In [21]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42
)


In [22]:
model = LogisticRegression(max_iter=1000)
model.fit(X_tr, y_tr)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000)

In [23]:
y_pred = model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))

Accuracy: 0.896540404040404
              precision    recall  f1-score   support

           0       0.96      0.94      0.95     22534
           1       0.79      0.69      0.73      3219
           2       0.82      0.91      0.87     12738
           3       0.70      0.39      0.50      1109

    accuracy                           0.90     39600
   macro avg       0.82      0.73      0.76     39600
weighted avg       0.90      0.90      0.89     39600



In [24]:
import pandas as pd

# 1. Load sample submission file (official ID list)
sample_df = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/Sample.csv')
                        
# 2. Check it
print(sample_df.head())
print(len(sample_df))

# 3. Predict on test data
test_predictions = model.predict(X_test)

# 4. Make sure lengths match
print(len(test_predictions), len(sample_df))

# 5. Copy sample and replace only label column
submission = sample_df.copy()
submission['label'] = test_predictions

# 6. Save file
submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully!")





   ID  label
0   1      0
1   2      0
2   3      0
3   4      0
4   5      0
102000
102000 102000
submission.csv created successfully!


In [25]:
submission['ID'].duplicated().sum()



np.int64(0)